In [ ]:
# Run 004 đã hoàn thành; không train lại để thay kết quả.
import subprocess,sys,os,importlib.util,importlib.metadata,platform
# Package discovery is cached after imports; changing pip packages cannot clear it safely.
loaded=[name for name in ('peft','torchao','chronos','transformers')
        if any(module==name or module.startswith(name+'.') for module in sys.modules)]
if loaded:
    raise RuntimeError('Restart Session rồi Run All trước khi cài thư viện: '+', '.join(loaded))
os.environ['HF_HUB_DISABLE_XET']='1'
# This lab uses ordinary float32 weights, not torchao quantization.
subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','torchao'])
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'chronos-forecasting==2.3.2','peft==0.21.2','accelerate>=1.1,<2',
    'datasets>=3,<5','pyarrow>=17,<27','scikit-learn>=1.5,<2'])
importlib.invalidate_caches()
if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError('torchao vẫn còn sau khi gỡ; kiểm tra pip output và Restart Session.')
import torch
from peft import LoraConfig,get_peft_model
from peft.import_utils import is_torchao_available
if is_torchao_available():
    raise RuntimeError('Lab yêu cầu PEFT chạy không có torchao.')
# Exercise the same adapter-injection path that previously failed, without a download.
smoke=get_peft_model(torch.nn.Sequential(torch.nn.Linear(4,4)),
                    LoraConfig(r=2,lora_alpha=4,target_modules=['0']))
smoke(torch.ones(1,4)).sum().backward()
if not any(p.requires_grad and p.grad is not None for p in smoke.parameters()):
    raise RuntimeError('LoRA preflight không tạo được gradient cho adapter.')
del smoke
LORA_ENVIRONMENT={'python':platform.python_version(),'torch':torch.__version__,
    'packages':{name:importlib.metadata.version(name) for name in
        ('chronos-forecasting','peft','transformers','accelerate','datasets')},
    'torchao_installed':False,'lora_injection_backward_ok':True}
LORA_PIP_FREEZE=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
print('LoRA preflight OK:',LORA_ENVIRONMENT)

In [ ]:
# Mặc định preflight; chỉ chuyển train sau khi kiểm tra evidence.
from pathlib import Path
import sys, subprocess, json, zipfile, shutil
from IPython.display import display, FileLink

MODE='preflight'
NOTEBOOK_URL=''  # Khi train: URL thật https://www.kaggle.com/code/<user>/<notebook>
SOURCE_COMMIT='bb47737b5bc400169a8bbd75f8c7534503ba18d5'
WORK=Path('/kaggle/working/quantstream_run004')
WORK.mkdir(parents=True,exist_ok=True)
CODE=WORK/('source-'+SOURCE_COMMIT[:12])
SNAPSHOT=WORK/'snapshot'
PREFLIGHT=WORK/'preflight'
OUTPUT=WORK/'run-004-seed-stability'
LOCK=PREFLIGHT/'candidate-lock.json'
if MODE not in {'preflight','train'}:
    raise ValueError('MODE phải là preflight hoặc train')
print('Mode:',MODE,'Source:',SOURCE_COMMIT)

In [ ]:
# Giữ source, snapshot và candidate lock đúng SHA.
if not CODE.exists():
    subprocess.check_call(['git','init','-q',str(CODE)])
    subprocess.check_call(['git','-C',str(CODE),'remote','add','origin',
                           'https://github.com/taiiswibu/quantstream-crypto-research.git'])
    subprocess.check_call(['git','-C',str(CODE),'fetch','-q','--depth','1','origin',SOURCE_COMMIT])
    subprocess.check_call(['git','-C',str(CODE),'checkout','-q','--detach','FETCH_HEAD'])
actual=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
if actual!=SOURCE_COMMIT:
    raise ValueError('Source khác SHA đã khóa; giữ output để kiểm tra')
if subprocess.check_output(['git','-C',str(CODE),'status','--porcelain'],text=True).strip():
    raise ValueError('Source working tree phải sạch')
if any(name=='quantstream' or name.startswith('quantstream.') for name in sys.modules):
    raise RuntimeError('Restart Session để không dùng nhầm quantstream đã import')
sys.path.insert(0,str(CODE))

INPUT=Path('/kaggle/input')
archives=list(INPUT.rglob('QuantStream_Kaggle_Input.zip'))
if len(archives)>1:
    raise ValueError('Chỉ Add Input một phiên bản dataset Run 003')
if not SNAPSHOT.exists():
    if archives:
        with zipfile.ZipFile(archives[0]) as archive:
            for info in archive.infolist():
                name=Path(info.filename)
                if name.is_absolute() or '..' in name.parts:
                    raise ValueError('Unsafe ZIP path')
                if name.parts and name.parts[0]=='snapshot':
                    archive.extract(info,WORK)
    else:
        manifests=[p for p in INPUT.rglob('manifest.json')
                   if (p.parent/'development.parquet').exists()]
        if len(manifests)!=1:
            raise FileNotFoundError('Add Input đúng dataset cũ đã chạy Run 003')
        shutil.copytree(manifests[0].parent,SNAPSHOT)
from quantstream.seed_lab import preflight_run004,run_seed_lab,archive_evidence,verify_run003_snapshot,SNAPSHOT_SHA
verified=verify_run003_snapshot(SNAPSHOT)
print('Snapshot:',verified['manifest_relation'])
print('Manifest hiện tại:',verified['manifest_sha256'])
print('Manifest Run 003:',verified['reference_manifest_sha256'])
print('Bốn file dữ liệu và cấu hình khớp Run 003. Chưa chạy research fit.')

In [ ]:
# Preflight smoke không tính là research fit; lưu cả evidence khi lỗi.
target=PREFLIGHT if MODE=='preflight' else OUTPUT
filename='QuantStream_Run004_Preflight.zip' if MODE=='preflight' else 'QuantStream_Run004_Output.zip'
archive=WORK.parent/filename
# Refuse before entering finally, so a repeated click cannot overwrite a previous archive.
if target.exists() or archive.exists():
    raise FileExistsError('Đã có evidence. Tải ZIP cũ và gửi kiểm tra, không xóa để chạy lại.')
try:
    if MODE=='preflight':
        result=preflight_run004(SNAPSHOT,PREFLIGHT)
        print('Preflight hoàn tất. Research fits đã chạy: 0')
        print('Tải QuantStream_Run004_Preflight.zip và gửi lại trước khi train.')
    else:
        result=run_seed_lab(SNAPSHOT,OUTPUT,LOCK,NOTEBOOK_URL)
        import pandas as pd
        display(pd.DataFrame(result['metrics']))
        print('Deployment:',result['deployment_status'])
finally:
    saved=archive_evidence(target,archive)
    if saved:
        display(FileLink(saved))

In [ ]:
# Preflight chưa có kết quả research; chỉ vẽ khi train hoàn tất.
if MODE == 'train' and 'metrics' in result:
    import pandas as pd
    import matplotlib.pyplot as plt
    seed_metrics = pd.DataFrame(result['metrics'])
    display(seed_metrics)
    metric_columns = [c for c in seed_metrics.columns if 'mae' in c.lower() and pd.api.types.is_numeric_dtype(seed_metrics[c])]
    if metric_columns:
        ax = seed_metrics[metric_columns].plot.bar(figsize=(10, 4), color=['#64748b', '#0f766e', '#2563eb'])
        ax.set(title='Run 004: all reported seed MAE', xlabel='Metric row', ylabel='MAE (return)')
        ax.grid(axis='y', alpha=.2); plt.tight_layout(); plt.show()
else:
    print('Preflight: chưa có metric research để vẽ.')